# Segmentation workflow for multichannel 3D images

OK. So we have seen how to segment and count objects using scikit-image and cellpose. Now we approach a new challenge; segmenting 3D objects. 

The general pipeline looks similar to what we saw earlier: ????????????????
1. Define object and constraints
2. Preprocess (illumination, noise, contrast)
    - with optional downsampling to reduce compute requirements when developing code.
3. Extract foreground
4. Generate seeds (if instances touch)
5. Separate objects (watershed)
6. Filter and correct
7. Quality-check and iterate

We segment using StarDist or Cellpose and have optional post-processing to remove objects, perform smoothing etc. Then we will counts the number of nuclei/cells per 3D image.

For quality control, we produce some visual outputs e.g. ????

In the next notebook, we will identify cell size and count subcellular structures in a multichannel analysis.

1. Define object and constraints
What to record up front
Object: we have in different channels nuclei, whole cells and sub-objects. Here, we segment and count nuclei.

Imaging: fluorescence

Size range: expected pixel area or diameter. Calculate:

pixel size = 512 / 246 (from metadata) = 2um
expected_pixel_diameter = real_diameter / pixel_diameter = 15 (from google search) / 2 = 7.5 pixels

Touching frequency: low (for nuclei)

Shape: round

Signal reliability: consistent

Error tolerance: more acceptable to miss objects or to over-split?

In [ ]:
## Libraries
## Load packages
import skimage
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pandas as pd
import os
import tifffile


# some more specific functions
from skimage.exposure import rescale_intensity
from skimage.transform import resize
from skimage.color import label2rgb

from cellpose import models

### How many images to segment? More takes longer but gives more info
num_images_to_segment = 5

## set dirs
cwd = Path.cwd()

# Set the main project directory by looking upwards for repo name "UCL-Biosciences-Image-Analysis"
# Find the first parent that matches the target folder
for p in cwd.parents:
    if p.name == "UCL-Biosciences-Image-Analysis":
        DIR = p
        break
else:
    raise FileNotFoundError("Base directory 'UCL-Biosciences-Image-Analysis' not found in path.")

## Set key variables
input_folder = DIR / "input_data" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF"
output_folder = DIR / "output" / "S-BIAD1272_30min_stimulation" / "240109_240110_S1_30min_pMAPK_EGF"
Path(output_folder).mkdir(parents=True, exist_ok=True)

### Input data setup
We specify the **input folder** containing raw images above. We first list all files and find out which channel contains the nuclei that we will segment and count in this notebook.

We then read in all images, keeping only the channel with the nuclei.

In [ ]:
# 1. load images and
# enter voxel size of your images
voxel_size_um = [1, 0.48, 0.48 ] # Z, Y, X; from metadata

# loading images from the input folder directory
# glob() finds files that match a pattern, in this case files that end with .tif
files = sorted(input_folder.glob("*.tif"))[: num_images_to_segment] # only the number of files we decided earlier

# 3d images with multiple channels have four dimensions:
# Z, X, Y and channel.
# For this analysis,  just want nuclei.
# So need to find out which dimension is channels and which channel has nuclei

# first work out the default order of the data in one file, f
f = files[0]
img = tifffile.imread(f)  # read in the tiff
img.shape # print its shape

# we see (16, 3, 512, 512)
# We know we have 3 channels, 512 x 512 2D images, and 16 2D images in the Z stack


In [ ]:
# check the channel order using the ~middle slice
mid_slice_idx = 8

mid_slice = img[ mid_slice_idx, ]

## check we have taken the right dimension. should show 3, 512, 512
mid_slice.shape


In [ ]:
# one plot per channel
n_channels = mid_slice.shape[0]

# empty plot grid with one row and n columns
fig, axs = plt.subplots(1, n_channels, figsize = (15, 5))

# loop through the channels and show the middle slice of each
for i in range(n_channels):
    axs[i].imshow(mid_slice[i])
    axs[i].axis("off")

## nuclei in first channel! We'll extract that for all images in the next cell

In [ ]:
# empty object for storing files
all_images = []

# loop through the files we found earlier (glob(".tif"))
for f in files:
    # read in the image
    img = tifffile.imread(f)  # starts in Z, C, Y, X
    
    # we want all Z, first channel, and all X and Y
    img_nucl = img[:, 0, :, :]  # output shape: (16, 512, 512)
    
    # append the nuclei image and filename
    all_images.append({
        "filename": f.stem, 
        "img": img_nucl
    })

# preview loaded images: first image [0] as example
# note we will use this syntax in loops, below
print(all_images[0]["filename"])       # e.g., "sample01.tif"
print(all_images[0]["img"].shape)  # (Z, Y, X)

# looks good? Now we are ready for segmentation!

### Nuclei preprocessing, segmentation, and quantification in 3D
This section describes the preprocessing, segmentation, and quantification of nuclei in 3D microscopy images. The workflow uses  **Cellpose** for segmentation, with post-processing and quality control steps available.

 1. **Preprocessing**: rescale to 0-1, optionally downsample.  

 2. **Segmentation**
  - Segmentation model for nuclei:
    - `Cellpose` (SAM-based segmentation, preferrably should be run on GPU)    
  - Output: **labelled 3D mask** where each nucleus is assigned a unique label.  

 3. **Quality Control (QC)**
- Save `.tif` mask stack and `.png` overlays (maximum-intensity projections with labels).  
- View some masks in the cell output to quickly check performance.


In [ ]:
# First we set some arguments
# if you don't want to downsize, set to 1
downsize_factor = 1 #

# whether to run cellpose with GPU, if not, will be slow.
gpu= True

In [ ]:
# Empty dictionary for storing output
results_dict = {}

# # loop over loaded images for segmentation
for i in range(len(all_images)):
    
    v = all_images[i]["img"]
    file_name = all_images[i]["filename"]
    
    print('starting for img:', file_name)
    
    # --- 2.1 nuclei preprocessing ---
    # convert to float32 - required for processing
    img_3d = v.astype('float32')

    # rescale the intensity to between 0 and 1
    img_3d = rescale_intensity(img_3d, out_range=(0, 1))

    # might need isotropic rescaling?

    # downsample to reduce computation
    z, y, x = img_3d.shape
    new_y = int(round(y * downsize_factor))
    new_x = int(round(x * downsize_factor))

    img_resized = resize(
        img_3d,
        (z, new_y, new_x),
        # order=1,
        # anti_aliasing=True,
        # preserve_range=True
    ).astype(img_3d.dtype)    
    
    # --- 2.2 nuclei segmentation ---

    print("Running Cellpose...")
    model = models.CellposeModel(gpu=gpu)

    mask, flows, styles = model.eval(
    img_3d,
    # diameter=diameter,      # diameter can be adjusted fpr better targeted segmentation?
    z_axis=0,               # specify for Cellpose assuming (Z, Y, X)
    do_3D=True              # enables 3D segmentation
    )

    results_dict[file_name] = mask
    
    ### Save output
    experiment_label=f"{file_name}_nuclei"

    ## Main segmented mask
    # first convert into 16 bit
    mask_uint16 = mask.astype(np.uint16)
    
    # save mask stack with experiment label in filename
    mask_path = output_folder / f"{experiment_label}_mask.tif"
    tifffile.imwrite(mask_path, mask_uint16)

    ## An overlay of segmentation on original
    # maximum intensity projections (MIPs) collapse a 3D volume into a 2D image
    # by taking the maximum pixel value along the Z axis at each (Y, X) position
    # we overlay on raw images - useful for a quick sanity check that your segmentation lines up with the raw signal.

    # MIP of raw
    mip_raw = np.max(v, axis=0)
    # normalise raw MIP for display
    mip_raw_float = mip_raw.astype(np.float32)
    mip_raw_float = (mip_raw_float - mip_raw_float.min()) / (
        mip_raw_float.max() - mip_raw_float.min() + 1e-8
    )

    # MIP of mask
    mip_mask = np.max(mask, axis=0)

    # overlay labels on raw MIP
    mip_overlay = label2rgb(mip_mask, image=mip_raw_float, bg_label=0, alpha=0.4)

    overlay_path = output_folder / f"{experiment_label}_overlay_MIP.png"
    plt.imsave(overlay_path, (mip_overlay * 255).astype(np.uint8))
    plt.close()


### Visualise Segmentation
If you get this far, that means Cellpose has finished and has done some kind of segmentation on your images.

Next, let's have a look at the output to make sure it is sensible.

In [ ]:
### View some masks - are they sensible?
# change which mask by editing img_to_plot
img_to_plot = 4

# take out the image and file name
vol = all_images[img_to_plot]["img"]
file_name = all_images[img_to_plot]["filename"]

# and find the mask
mask = results_dict[file_name]

## look at the middle slice
img = vol[8]
mask_slice = mask[8]

## plot
# if only 1 plot, can do subplots(1, 1)
fig, axes = plt.subplots(1, 1, figsize=(5* 1, 5))

axes.imshow(img, cmap="gray") # grey for original

axes.imshow(np.ma.masked_where(mask_slice == 0, mask_slice), # don't show where mask slice == 0
          cmap="autumn", alpha=0.5) # but plot the rest with autumn colour palette

axes.set_title(file_name)
axes.axis("off")

plt.tight_layout()
plt.show()

 4. **Quantification** (optional)
- Extract per-object features such as `count` and `volume`.  
- Compare measured nucleus volumes against expected biological range (e.g. 10 µm diameter sphere).  

In [ ]:
from skimage.morphology import remove_small_holes, remove_small_objects
from skimage.morphology import ball, binary_closing
from skimage.measure import label, regionprops_table


# loop over loaded images for segmentation
for img_to_plot in range(len(all_images)):
    
    vol = all_images[img_to_plot]["img"]
    file_name = all_images[img_to_plot]["filename"]
    mask = results_dict[file_name]

    # tidy mask for counting
    # ensure binary for morphology
    binary = mask > 0

    ## fill in small holes - requires a "ball" argument to define the volume of space filled
    # footprint is the object we create (the space to fill)
    footprint  = ball(1)
    binary = binary_closing(binary, footprint )
    
    # we can also use built in functions to remove small holes and objects
    binary = remove_small_holes(binary, area_threshold=64)
    binary = remove_small_objects(binary, min_size=50)
    
    # relabel to instances
    mask = label(binary)

    # --- quantify nuclei ---
    # regionprops_table pulls out key info from the mask
    # properties can include area, centroid and intensity
    props = ["area"]
    props_table = regionprops_table(
        mask,
        intensity_image=intensity_img if "intensity" in props else None, # need an intensity image to quantify intensity per object
        properties= props# we just look at area
    )
    
    df = pd.DataFrame(props_table)
    
    # area is a voxel count, which we need to convert to volume
    
    # use the size of the voxels in um
    voxel_size_um = (1, 0.48, 0.48) # known from metadata
    
    # Multiplying the voxel count gives you total volume per voxel in µm³
    voxel_vol = np.prod(voxel_size_um) if voxel_size_um is not None else 1
    
    # now multiple the area (voxel count) by the volume per voxel
    df["volume"] = df["area"] * voxel_vol
    # and remove area column
    df.drop(columns=["area"], inplace=True)
    
    ## pull out some information and print!
    summary = {}
    summary["file_name"] = file_name
    summary["object_count"] = len(df)
    summary["mean_volume"] = df["volume"].mean().round() if len(df) > 0 else 0
    
    print(summary)


### Summary
Here we have processed, segmented and counted nuclei in 3D, multichannel images! Next, we will see how we can analyse all three channels and combine the results to get the number of subcellular structures per cell in each image!